In [0]:
# GOLD LAYER — IMPERATIVE APPROACH
# cell 1 Imports and configurations
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
#spark_round  é a função do Spark que arredonda colunas.

#source - silver imperative (ADLS paths)
STORAGE_ACCOUNT = "marketpulsedatalake"
SILVER_FACT_PRICES_PATH = (    f"abfss://silver@{STORAGE_ACCOUNT}.dfs.core.windows.net/fact_prices/"
)
#target - gold ( new container)
GOLD_DAILY_SUMMERY_PATH = (    f"abfss://gold@{STORAGE_ACCOUNT}.dfs.core.windows.net/daily_summary/"
)

print("✅ Configuration loaded")

In [0]:
 def read_silver_fact_prices() -> DataFrame:
     """Reads the Silver fact_prices Delta table.
    
    Returns:
        DataFrame with stock prices: symbol, trade_date, OHLCV."""

    return spark.read.format("delta").load(SILVER_FACT_PRICES_PATH)


def build_daily_summary(df_fact: DataFrame) -> DataFrame:
    """
    Builds daily summary metrics per stock and trading day.
    
    Returns:
        DataFrame with OHLCV + return %, range, range %.
    """
    return(
        df_fact
            .withColumn(
                "daily_return_pct",
                F.round(((F.col("close") - F.col("open")) / F.col("open"))*100, 2)
            )
            .withColumn(
                "intraday_range",
                F.round(F.col("high") - F.col("low"),2)
            )
            .withColumn(
                "intraday_range_pct",
                F.round(((F.col("high") - F.col("low")) / F.col("open"))*100,2)
            )
            .withColumn(
                "ingest_timestamp",
                F.current_timestamp()
            )
     
    )
    

